# FIAP Global Solution 2026.2, CitizenTech
## Golpe Zero - Fundamentos: Implementação de uma LLM do zero
**Disciplina:** Generative AI for Engineering - Prof. Francisco Elânio Bezerra

---
### Objetivo do caderno
Demonstrar a mecânica fundamental de modelos de linguagem sem o uso de bibliotecas de alto nível (PyTorch, TensorFlow, Hugging Face ou Scikit-Learn).

Aqui construímos a espinha dorsal matemática de uma rede neural autoregressiva usando **exclusivamente NumPy**:
1. **Tokenização Manual:** Criação do vocabulário e mapeamento bidirecional de IDs (`palavra_para_id` e `id_para_palavra`).
2. **Tabela de Embeddings ($E$):** Inicialização de vetores densos aleatórios de dimensão $D=4$.
3. **Context Pooling:** Fusão do contexto de entrada por média aritmética.
4. **Projeção Linear ($W$):** Matriz de pesos que mapeia o espaço latente de volta ao tamanho do vocabulário ($V=14$).
5. **Logits & Softmax Estável:** Cálculo das pontuações brutas e normalização probabilística via constante de Euler ($e^x$).
6. **Cross-Entropy Loss:** Medição da divergência em relação ao alvo (One-Hot implícito).
7. **Backpropagation Manual:** Cálculo dos gradientes analíticos grad_W e grad_E via produto externo e transposta.
8. **Loop de Otimização (SGD):** Treinamento por Gradiente Descendente Estocástico até a convergência.

### 1. Dados de Treinamento e Construção do Vocabulário (Tokenização)

In [1]:
import numpy as np
np.random.seed(42)

# Pares de treinamento: (Contexto de Entrada -> Palavra Alvo)
pares_treino = [
    ("como tu tá", "bem"),
    ("manda um pix", "golpe"),
    ("corta o cartão", "golpe"),
    ("bom dia vó", "bem"),
]

# Extração de todas as palavras únicas de forma legível e determinística
sequencia_palavras = []
for pergunta, resposta in pares_treino:
    for p in pergunta.split():
        if p not in sequencia_palavras:
            sequencia_palavras.append(p)
    for r in resposta.split():
        if r not in sequencia_palavras:
            sequencia_palavras.append(r)

# Dicionários de consulta direta O(1)
palavra_para_id = {palavra: i for i, palavra in enumerate(sequencia_palavras)}
id_para_palavra = {i: palavra for i, palavra in enumerate(sequencia_palavras)}

V = len(sequencia_palavras) # Tamanho do Vocabulário (14 palavras)
D = 4                       # Dimensões de cada vetor de embedding

print(f"Vocabulário ({V} tokens):", sequencia_palavras)
print("Mapeamento Token -> ID:", palavra_para_id)

Vocabulário (14 tokens): ['como', 'tu', 'tá', 'bem', 'manda', 'um', 'pix', 'golpe', 'corta', 'o', 'cartão', 'bom', 'dia', 'vó']
Mapeamento Token -> ID: {'como': 0, 'tu': 1, 'tá': 2, 'bem': 3, 'manda': 4, 'um': 5, 'pix': 6, 'golpe': 7, 'corta': 8, 'o': 9, 'cartão': 10, 'bom': 11, 'dia': 12, 'vó': 13}


### 2. Inicialização dos Parâmetros (E e W)

In [2]:
# Tabela de Embeddings de Entrada (V x D): cada palavra tem 4 coordenadas latentes
E = np.random.randn(V, D) * 0.1

# Matriz de Pesos de Saída (D x V): requisitos para cada uma das 14 palavras concorrentes
W = np.random.randn(D, V) * 0.1

print(f"Formato da Tabela de Embeddings E : {E.shape} -> (Tokens, Dimensões)")
print(f"Formato da Matriz de Pesos W       : {W.shape} -> (Dimensões, Tokens)")

# Exemplo do embedding inicial aleatório da palavra 'pix':
print("Embedding inicial de 'pix':", np.round(E[palavra_para_id["pix"]], 4))

Formato da Tabela de Embeddings E : (14, 4) -> (Tokens, Dimensões)
Formato da Matriz de Pesos W       : (4, 14) -> (Dimensões, Tokens)
Embedding inicial de 'pix': [-0.0544  0.0111 -0.1151  0.0376]


### 3. Funções Matemáticas do Motor (Forward Pass, Softmax e Loss)

In [3]:
def softmax(logits: np.ndarray) -> np.ndarray:
    """Converte notas brutas em probabilidades contidas entre 0 e 1 (soma 100%).
    A subtração do máximo (logits - max) previne estouro numérico (overflow).
    """
    exp_ajustado = np.exp(logits - np.max(logits))
    return exp_ajustado / np.sum(exp_ajustado)

def predizer(contexto: str, E_atual: np.ndarray, W_atual: np.ndarray):
    """Executa o Forward Pass para um contexto textual."""
    ids = [palavra_para_id[p] for p in contexto.split() if p in palavra_para_id]
    # Mean Pooling dos embeddings do contexto
    h = np.mean(E_atual[ids], axis=0)
    # Produto Escalar (dot) contra as 14 colunas de W
    logits = np.dot(h, W_atual)
    probs = softmax(logits)
    id_chute = np.argmax(logits)
    return id_chute, id_para_palavra[id_chute], probs

# Teste com a rede ainda não treinada (pesos aleatórios):
id_c, chute_inicial, probs_iniciais = predizer("manda um", E, W)
prob_pix_inicial = probs_iniciais[palavra_para_id["pix"]]
print(f"Contexto: 'manda um' | Chute Inicial: '{chute_inicial}' | Certeza no Pix: {prob_pix_inicial:.2%}")

Contexto: 'manda um' | Chute Inicial: 'dia' | Certeza no Pix: 7.12%


### 4. Treinamento Manual com Backpropagation (Gradiente Descendente)

In [4]:
epocas = 200
lr = 0.3 # Learning rate (tamanho do passo de descida do gradiente)

historico_loss = []

print("=" * 75)
print(f"{'Época':<8} | {'Loss Médio':<12} | {'Acurácia Treino':<16} | Exemplo ('manda um')")
print("=" * 75)

for epoca in range(1, epocas + 1):
    loss_epoca = 0.0
    acertos = 0
    
    for entrada, alvo in pares_treino:
        ids_entrada = [palavra_para_id[p] for p in entrada.split()]
        id_alvo = palavra_para_id[alvo]
        
        # 1. FORWARD PASS
        h = np.mean(E[ids_entrada], axis=0) # Vetor latente do contexto
        logits = np.dot(h, W)               # 14 notas brutas
        probs = softmax(logits)             # Distribuição de probabilidade
        
        # 2. CÁLCULO DA PERDA (Cross-Entropy Loss)
        loss_epoca += -np.log(probs[id_alvo] + 1e-9)
        if np.argmax(probs) == id_alvo:
            acertos += 1
            
        # 3. CÁLCULO DO ERRO ANALÍTICO (dLoss/dLogits)
        erro = probs.copy()
        erro[id_alvo] -= 1.0 # Subtrai 1.0 da posição do alvo verdadeiro
        
        # 4. BACKPROPAGATION NA UNHA
        grad_W = np.outer(h, erro)    # Derivada em relação aos pesos de saída (D x V)
        grad_h = np.dot(erro, W.T)    # Derivada em relação aos embeddings de entrada (D)
        
        # 5. ATUALIZAÇÃO DOS PARÂMETROS (SGD)
        W -= lr * grad_W
        for idx in ids_entrada:
            E[idx] -= lr * (grad_h / len(ids_entrada))
            
    loss_medio = loss_epoca / len(pares_treino)
    acuracia = (acertos / len(pares_treino)) * 100
    historico_loss.append(loss_medio)
    
    if epoca == 1 or epoca % 25 == 0:
        _, chute_manda, _ = predizer("manda um", E, W)
        print(f"{epoca:<8} | {loss_medio:<12.4f} | {acuracia:<15.1f}% | Chutou: '{chute_manda}'")

print("=" * 75)

Época    | Loss Médio   | Acurácia Treino  | Exemplo ('manda um')
1        | 2.6461       | 0.0            % | Chutou: 'dia'
25       | 0.0508       | 100.0          % | Chutou: 'golpe'
50       | 0.0100       | 100.0          % | Chutou: 'golpe'
75       | 0.0052       | 100.0          % | Chutou: 'golpe'
100      | 0.0034       | 100.0          % | Chutou: 'golpe'
125      | 0.0025       | 100.0          % | Chutou: 'golpe'
150      | 0.0020       | 100.0          % | Chutou: 'golpe'
175      | 0.0016       | 100.0          % | Chutou: 'golpe'
200      | 0.0014       | 100.0          % | Chutou: 'golpe'


### 5. Avaliação Final: Verificação da Convergência e dos Novos Embeddings

In [5]:
print("--- RESULTADOS APÓS O TREINAMENTO ---\n")
for entrada, alvo in pares_treino:
    id_c, chute, probs = predizer(entrada, E, W)
    prob_certa = probs[palavra_para_id[alvo]]
    status = "CORRETO" if chute == alvo else "ERRADO"
    print(f"Entrada: {entrada:<16} | Chute: {chute:<8} | Alvo: {alvo:<8} | Certeza: {prob_certa:.2%} [{status}]")

print("\n--- EMBEDDINGS APRENDIDOS (Agrupamento Semântico Espontâneo) ---")
print("Vetor de 'pix'   :", np.round(E[palavra_para_id["pix"]], 3))
print("Vetor de 'cartão':", np.round(E[palavra_para_id["cartão"]], 3))
print("Vetor de 'vó'    :", np.round(E[palavra_para_id["vó"]], 3))

# Similaridade de cosseno manual entre os embeddings aprendidos:
def sim_cos(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

cos_pix_cartao = sim_cos(E[palavra_para_id["pix"]], E[palavra_para_id["cartão"]])
cos_pix_vo = sim_cos(E[palavra_para_id["pix"]], E[palavra_para_id["vó"]])

print(f"\nSimilaridade Cosseno entre 'pix' e 'cartão' (ambos contexto de golpe) : {cos_pix_cartao:.4f}")
print(f"Similaridade Cosseno entre 'pix' e 'vó' (contextos opostos)            : {cos_pix_vo:.4f}")

--- RESULTADOS APÓS O TREINAMENTO ---

Entrada: como tu tá       | Chute: bem      | Alvo: bem      | Certeza: 99.86% [CORRETO]
Entrada: manda um pix     | Chute: golpe    | Alvo: golpe    | Certeza: 99.86% [CORRETO]
Entrada: corta o cartão   | Chute: golpe    | Alvo: golpe    | Certeza: 99.86% [CORRETO]
Entrada: bom dia vó       | Chute: bem      | Alvo: bem      | Certeza: 99.86% [CORRETO]

--- EMBEDDINGS APRENDIDOS (Agrupamento Semântico Espontâneo) ---
Vetor de 'pix'   : [-0.936 -1.313 -0.167 -0.956]
Vetor de 'cartão': [-0.798 -1.268 -0.077 -1.02 ]
Vetor de 'vó'    : [0.139 0.917 1.437 1.127]

Similaridade Cosseno entre 'pix' e 'cartão' (ambos contexto de golpe) : 0.9958
Similaridade Cosseno entre 'pix' e 'vó' (contextos opostos)            : -0.6881
